*0.4 Deep learning basics*

# Transformer from scratch

**The situation.** The pieces are on the table: positional encoding, multi-head attention, encoder layers, a causal decoder. The 2017 paper put them together as an encoder-decoder for translation. The date-normalisation task from items 8–9 is a small translation — a fair test of the full machine against the seq2seq models.

**The full transformer.** Encoder stack reads the input. Decoder stack, with causal self-attention *and* cross-attention to the encoder outputs, writes the output. Built here from the same PyTorch modules production code uses (`nn.Transformer`), with the embedding, positions and output head written out.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import random
from datetime import date, timedelta

import torch

random.seed(0)
MONTHS = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]


def random_pair():
    day = date(2000, 1, 1) + timedelta(days=random.randint(0, 365 * 30))
    formats = [
        f"{MONTHS[day.month - 1]} {day.day}, {day.year}",
        f"{day.day}/{day.month}/{day.year % 100:02d}",
        f"{day.day} {MONTHS[day.month - 1][:3]} {day.year}",
        f"{day.month:02d}-{day.day:02d}-{day.year}",
        f"{day.day}th of {MONTHS[day.month - 1]} {day.year}",
    ]
    return random.choice(formats), day.isoformat()


pairs = []
for _ in range(4000):
    pairs.append(random_pair())
all_text = ""
for source_text, target_text in pairs:
    all_text += source_text + target_text
alphabet = ["<pad>", "<start>", "<end>"] + sorted(set(all_text))
index = {}
for position, character in enumerate(alphabet):
    index[character] = position
PAD, START, END = 0, 1, 2


def encode(text, length):
    ids = []
    for character in text[:length]:
        ids.append(index[character])
    return ids + [PAD] * (length - len(ids))


IN_LEN, OUT_LEN = 24, 12
input_rows = []
target_rows = []
for source_text, target_text in pairs:
    input_rows.append(encode(source_text, IN_LEN))
    target_rows.append([START] + encode(target_text, OUT_LEN - 2) + [END])
inputs = torch.tensor(input_rows)
targets = torch.tensor(target_rows)
train_inputs, val_inputs = inputs[:3600], inputs[3600:]
train_targets, val_targets = targets[:3600], targets[3600:]
print("input", tuple(inputs.shape), "| target", tuple(targets.shape))

input (4000, 24) | target (4000, 12)


In [3]:
import time

import torch.nn.functional as F
from torch import nn


def count_parameters(module):
    total = 0
    for parameter in module.parameters():
        total += parameter.numel()
    return total


import warnings

warnings.filterwarnings("ignore", category=UserWarning)


class DateTransformer(nn.Module):
    def __init__(self, alphabet_size, d_model=64, heads=4, layers=2):
        super().__init__()
        self.embedding = nn.Embedding(alphabet_size, d_model, padding_idx=PAD)
        self.positions = nn.Embedding(IN_LEN, d_model)
        self.transformer = nn.Transformer(
            d_model,
            heads,
            num_encoder_layers=layers,
            num_decoder_layers=layers,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True,
        )
        self.output = nn.Linear(d_model, alphabet_size)

    def embed(self, ids):
        return self.embedding(ids) + self.positions(torch.arange(ids.shape[1]))

    def forward(self, source, target_in):
        causal = nn.Transformer.generate_square_subsequent_mask(target_in.shape[1])
        decoded = self.transformer(
            self.embed(source),
            self.embed(target_in),
            tgt_mask=causal,
            src_key_padding_mask=source == PAD,
            tgt_key_padding_mask=target_in == PAD,
        )
        return self.output(decoded)

    @torch.no_grad()
    def predict(self, source):
        memory = self.transformer.encoder(
            self.embed(source), src_key_padding_mask=source == PAD
        )  # encode once
        out = torch.full((source.shape[0], 1), START)
        for _ in range(OUT_LEN - 1):
            causal = nn.Transformer.generate_square_subsequent_mask(out.shape[1])
            decoded = self.transformer.decoder(
                self.embed(out), memory, tgt_mask=causal, memory_key_padding_mask=source == PAD
            )
            next_token = self.output(decoded[:, -1]).argmax(
                dim=-1, keepdim=True
            )  # only the newest position
            out = torch.cat([out, next_token], dim=1)
        return out[:, 1:]


def exact_match(model, source, target):
    model.eval()
    return (model.predict(source) == target[:, 1:]).all(dim=1).float().mean().item()


def train(model, epochs=12, lr=1e-3, batch_size=64):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(train_inputs))
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            logits = model(train_inputs[batch], train_targets[batch][:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                train_targets[batch][:, 1:].reshape(-1),
                ignore_index=PAD,
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epoch % 3 == 0:
            print(
                
                    f"epoch {epoch:>2}  loss {loss.item():.3f}  exact-match on validation "
                    f"{exact_match(model, val_inputs, val_targets):.1%}"
                
            )
    return exact_match(model, val_inputs, val_targets)


torch.manual_seed(0)
started = time.perf_counter()
model = DateTransformer(len(alphabet))
transformer_accuracy = train(model)
print(f"trained in {time.perf_counter() - started:.0f} s | parameters: {count_parameters(model):,}")
assert transformer_accuracy > 0.8

epoch  3  loss 0.406  exact-match on validation 7.0%


epoch  6  loss 0.039  exact-match on validation 79.5%


epoch  9  loss 0.038  exact-match on validation 90.0%


epoch 12  loss 0.010  exact-match on validation 94.0%
trained in 23 s | parameters: 174,892


**Reading the output.** The full transformer reaches a high exact-match on the date task, comparable to or better than seq2seq + attention, and every encoder position was processed in parallel. Note what `predict` does: encode once, then decode one character at a time, each time looking only at the newest position — the shape of every LLM's generation loop.

```
"March 3, 2024" ─▶ embed + positions ─▶ ENCODER (self-attention × 2) ─▶ memory
                                                                          │ cross-attention
"<start> 2 0 2" ─▶ embed + positions ─▶ DECODER (causal self-attn × 2) ◀──┘ ─▶ next char "4"
```

**The rule to remember.** Encoder reads, decoder writes, cross-attention connects them, causal mask keeps the decoder honest. Every piece is one of the previous four items.

| Use it when | Don't when | Instead use |
|---|---|---|
| translation-shaped tasks with an input and an output sequence; understanding the architecture | plain text generation | decoder-only (nanoGPT, next) |

**Watch out**
- Three masks: causal (decoder), source padding, target padding. Missing any one trains anyway and generates worse.
- `nn.Transformer` defaults are the 2017 paper's; modern models use pre-norm, RoPE, no biases — the same skeleton with better parts.
- `predict` re-runs the decoder over the whole prefix each step; the KV cache is the optimisation that avoids it.